# Persian AI-Text Detection — Colab retraining for JEV + ParsBERT

Companion to `colab_retrain.ipynb`, covering the two models that notebook doesn't
train:

- **ParsBERT** (`HooshvareLab/bert-fa-base-uncased`) — a bidirectional BERT-style
  encoder, fine-tuned as a plain binary classifier with `[CLS]` pooling. It is the
  fourth deep-learning model, added after the report's error analysis found BiLSTM
  (embeddings learned from scratch on ~21k rows) to be the model most exposed to
  distribution shift, and GPT-2 to be a *causal* decoder repurposed for
  classification rather than one built for it.
- **JEV "System One"** (`pytorch-jev`) — a decision model that doesn't generate an
  answer, it *answers typed questions* about a state in a single forward pass: a
  `choice` question (human vs AI, as a distribution), a `score` question
  (machine-likeness on a 5-level ordinal scale, answered as a probability-weighted
  value that can fall between levels) and a `noul` question (probability that a
  human must review). Trained on soft targets distilled from the deployed
  TF-IDF + Logistic Regression pipeline.

Both models get the full dataset, a GPU, mixed precision, and validation-based
early stopping — none of which the local CPU trainers could afford.

**Runtime:** `Runtime > Change runtime type > GPU` before running.

## What to put in Google Drive first

Create `MyDrive/AITextDetection/` and copy these four files out of your project
root into it:

| file | why |
| --- | --- |
| `processedDataset.csv` | the balanced 21,092-row dataset all the models train on |
| `tfidf_vectorizer.joblib` | the **deployed** vectorizer — JEV's teacher features must match it exactly |
| `logregModel.joblib` | the **deployed** logistic regression that JEV distils its soft targets from |
| `models.json` | read only to carry the current `ensembleWeight` values forward |

`tfidf_vectorizer.joblib` + `logregModel.joblib` are ~8 MB together, so Drive is
the sane place for them; `processedDataset.csv` is 160 MB and a browser
`files.upload()` of that tends to time out.

## What you get back

Everything is written to `MyDrive/AITextDetection/trained/` as soon as each model
finishes, so a runtime disconnect can't lose the work, and the last cell also zips
the whole set for a direct browser download:

- `parsbertClassifierModel.pt` + `parsbert_tokenizer/` — drop into the project root;
  `utils.py` rebuilds the encoder offline from the `backbone_config` carried inside
  the checkpoint and prefers the exported tokenizer folder, so no Hub access at
  startup.
- `jevModel.pt` + `gpt2_tokenizer/` — same offline-first story.
- `models_colab_update.json` — a `models.json` fragment using the exact ids
  (`parsbert`, `jev_system_one`) and `runtime` values (`pytorch-parsbert`,
  `pytorch-jev`) `utils.py` already dispatches on, so merging it needs no backend
  change.

In [ ]:
!pip install -q shekar "transformers>=4.40" scikit-learn joblib

import torch

print("torch:", torch.__version__, "| transformers:", __import__("transformers").__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU — go to Runtime > Change runtime type > GPU before continuing.")

In [ ]:
import json
import math
import multiprocessing
import os
import re
import shutil
import time
from collections import Counter

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from joblib import Parallel, delayed
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Edit this if you put the project files somewhere else in your Drive.
DRIVE_DIR = "/content/drive/MyDrive/AITextDetection"
OUT_DIR = os.path.join(DRIVE_DIR, "trained")
os.makedirs(OUT_DIR, exist_ok=True)

REQUIRED = {
    "processedDataset.csv": "the dataset every model trains on",
    "tfidf_vectorizer.joblib": "the deployed vectorizer JEV's teacher features must match",
    "logregModel.joblib": "the deployed logistic regression JEV distils soft targets from",
}
missing = [
    f"{name}  ({why})" for name, why in REQUIRED.items()
    if not os.path.isfile(os.path.join(DRIVE_DIR, name))
]
if missing:
    raise FileNotFoundError(
        "Copy these into " + DRIVE_DIR + ":\n  - " + "\n  - ".join(missing)
    )
print("Drive ready. Output ->", OUT_DIR)

## 2. Load the dataset

`processedDataset.csv` is the already balanced and shuffled dataset the deployed
models were trained on, so the numbers below are directly comparable to the ones
in `models.json`.

In [ ]:
df = pd.read_csv(os.path.join(DRIVE_DIR, "processedDataset.csv"))
print(df.shape)
print(df["aiGenerated"].value_counts())
df.head(3)

## 3. Train / validation / test split

Same seed and 80/20 stratified test split as `training/split.py`, so `test_idx`
lines up with the held-out set the currently-deployed models were scored on. The
20% test slice is touched **exactly once**, at the end, per model; the 10%
validation slice carved out of the training portion is what early stopping uses.

In [ ]:
TEST_SIZE = 0.2
VAL_SIZE = 0.1  # fraction of the train+val portion

idx = df.index.to_numpy()
y_all = df["aiGenerated"].to_numpy()

train_val_idx, test_idx = train_test_split(
    idx, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all
)
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=y_all[train_val_idx],
)

print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")
y_train, y_val, y_test = y_all[train_idx], y_all[val_idx], y_all[test_idx]

## 4. Normalization for the pretrained models

Both models below run on a pretrained tokenizer, so they get only shekar's
`Normalizer` — Arabic→Persian character unification, digit and punctuation
normalization, diacritic removal, repeated-letter and ZWNJ fixes. No
lemmatization, no word tokenization: that would strip the inflection and morphology
the models were pretrained on and hand them a token stream unlike anything in
their pretraining data. This mirrors `utils._normalize_for_gpt2`, which is exactly
what the server calls for these two runtimes, so train and serve see identical text.

Cached to Drive — the first run is the only one that pays for it.

In [ ]:
from shekar import Normalizer as ShekarNormalizer

_normalizer = ShekarNormalizer()


def normalize_for_gpt2(text: str) -> str:
    text = re.sub(r"\n", " ", str(text))
    return _normalizer(text)


NORM_CACHE = os.path.join(DRIVE_DIR, "normalized_texts_gpt2.joblib")

try:
    normalized_texts = joblib.load(NORM_CACHE)
    assert len(normalized_texts) == len(df)
    print(f"Loaded cached normalization for {len(normalized_texts)} documents.")
except (FileNotFoundError, AssertionError):
    print(f"Normalizing {len(df)} documents on {multiprocessing.cpu_count()} cores...")
    normalized_texts = Parallel(n_jobs=multiprocessing.cpu_count())(
        delayed(normalize_for_gpt2)(t) for t in df["text"]
    )
    joblib.dump(normalized_texts, NORM_CACHE)
    print("Cached to Drive for next time.")

## 5. ParsBERT — bidirectional encoder fine-tune

`HooshvareLab/bert-fa-base-uncased` is a 12-layer BERT pretrained on a large
Persian corpus, so it brings the same real language knowledge GPT-2 does, but with
an architecture purpose-built for sequence classification: every token attends to
the whole sequence and a dedicated `[CLS]` token summarizes it, instead of a causal
decoder whose only "saw everything" token is found by index arithmetic at the end.

The model class below must stay architecturally identical to
`utils.ParsBERTClassifier`; section 5d asserts that before anything is written.

In [ ]:
from transformers import AutoModel, AutoTokenizer

PARSBERT_MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"

parsbert_tokenizer = AutoTokenizer.from_pretrained(PARSBERT_MODEL_NAME)
parsbert_backbone = AutoModel.from_pretrained(PARSBERT_MODEL_NAME)
parsbert_hidden_size = parsbert_backbone.config.hidden_size
print(f"hidden_size={parsbert_hidden_size}  vocab_size={parsbert_backbone.config.vocab_size}")

In [ ]:
PARSBERT_SEQ_LEN = 256
PARSBERT_BATCH_SIZE = 32
PARSBERT_MAX_EPOCHS = 10
PARSBERT_PATIENCE = 2
PARSBERT_LR_BACKBONE = 2e-5
PARSBERT_LR_HEAD = 1e-3
PARSBERT_WEIGHT_DECAY = 0.01

# Every encoder layer + the pooler is fine-tuned; the (much larger) word-embedding
# table stays frozen, which lowers overfitting risk on 21k rows without hurting
# quality in practice.
for p in parsbert_backbone.parameters():
    p.requires_grad = False
for layer in parsbert_backbone.encoder.layer:
    for p in layer.parameters():
        p.requires_grad = True
for p in parsbert_backbone.pooler.parameters():
    p.requires_grad = True


class ParsBERTClassifier(nn.Module):
    # Must stay architecturally identical to utils.py's ParsBERTClassifier.

    def __init__(self, backbone, hidden_size):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        # [CLS] is always token position 0 for a BERT-style encoder. Unlike
        # GPT-2's causal attention, every position here already attends to the
        # whole sequence, so there's no attention-mask index arithmetic needed to
        # find "the token that saw everything".
        hidden = self.backbone(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        return self.head(hidden[:, 0]).squeeze(-1)


parsbert_model = ParsBERTClassifier(parsbert_backbone, parsbert_hidden_size).to(DEVICE)
parsbert_trainable = [p for p in parsbert_backbone.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in parsbert_trainable) + sum(
    p.numel() for p in parsbert_model.head.parameters()
)
print(f"Trainable parameters: {n_trainable:,}")

In [ ]:
def parsbert_encode(text_batch):
    enc = parsbert_tokenizer(
        list(text_batch),
        return_tensors="pt",
        truncation=True,
        max_length=PARSBERT_SEQ_LEN,
        padding="max_length",
    )
    return enc["input_ids"], enc["attention_mask"]


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i], self.labels[i]


def collate(batch):
    texts, labels = zip(*batch)
    input_ids, attention_mask = parsbert_encode(texts)
    return input_ids, attention_mask, torch.tensor(labels, dtype=torch.float32)


parsbert_train_texts = [normalized_texts[i] for i in train_idx]
parsbert_val_texts = [normalized_texts[i] for i in val_idx]
parsbert_test_texts = [normalized_texts[i] for i in test_idx]

parsbert_train_loader = DataLoader(
    TextDataset(parsbert_train_texts, y_train),
    batch_size=PARSBERT_BATCH_SIZE, shuffle=True, collate_fn=collate,
)
parsbert_val_loader = DataLoader(
    TextDataset(parsbert_val_texts, y_val), batch_size=64, shuffle=False, collate_fn=collate,
)
parsbert_test_loader = DataLoader(
    TextDataset(parsbert_test_texts, y_test), batch_size=64, shuffle=False, collate_fn=collate,
)
print(f"{len(parsbert_train_texts)} train / {len(parsbert_val_texts)} val / "
      f"{len(parsbert_test_texts)} test")

In [ ]:
parsbert_optimizer = torch.optim.AdamW(
    [
        {"params": parsbert_trainable, "lr": PARSBERT_LR_BACKBONE},
        {"params": parsbert_model.head.parameters(), "lr": PARSBERT_LR_HEAD},
    ],
    weight_decay=PARSBERT_WEIGHT_DECAY,
)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
AMP = DEVICE.type == "cuda"


def parsbert_eval(loader):
    parsbert_model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, yb in loader:
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                p = torch.sigmoid(parsbert_model(input_ids.to(DEVICE), attention_mask.to(DEVICE)))
            probs.append(p.float().cpu().numpy())
            labels.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(labels)


def parsbert_score(labels, probs):
    preds = (probs >= 0.5).astype(int)
    return f1_score(labels, preds), accuracy_score(labels, preds)


best_val_f1, best_state, stale = -1.0, None, 0

for epoch in range(1, PARSBERT_MAX_EPOCHS + 1):
    t0 = time.time()
    parsbert_model.train()
    total_loss = 0.0
    for i, (input_ids, attention_mask, yb) in enumerate(parsbert_train_loader, start=1):
        input_ids, attention_mask, yb = (
            input_ids.to(DEVICE), attention_mask.to(DEVICE), yb.to(DEVICE)
        )
        parsbert_optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            loss = criterion(parsbert_model(input_ids, attention_mask), yb)
        scaler.scale(loss).backward()
        scaler.unscale_(parsbert_optimizer)
        torch.nn.utils.clip_grad_norm_(parsbert_model.parameters(), max_norm=1.0)
        scaler.step(parsbert_optimizer)
        scaler.update()
        total_loss += loss.item() * len(yb)
        if i % 50 == 0:
            print(f"  epoch {epoch} batch {i}/{len(parsbert_train_loader)} "
                  f"loss={loss.item():.4f} elapsed={time.time() - t0:.0f}s", flush=True)

    val_probs, val_labels = parsbert_eval(parsbert_val_loader)
    val_f1, val_acc = parsbert_score(val_labels, val_probs)
    print(f"Epoch {epoch}/{PARSBERT_MAX_EPOCHS}  loss={total_loss / len(parsbert_train_texts):.4f}  "
          f"val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  ({time.time() - t0:.0f}s)")

    if val_f1 > best_val_f1:
        best_val_f1, stale = val_f1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in parsbert_model.state_dict().items()}
    else:
        stale += 1
        if stale >= PARSBERT_PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val improvement for {PARSBERT_PATIENCE} epochs).")
            break

parsbert_model.load_state_dict(best_state)
print(f"Best validation F1: {best_val_f1:.4f}")

In [ ]:
# Single held-out evaluation — the test slice is touched here and nowhere else.
probs, labels = parsbert_eval(parsbert_test_loader)
preds = (probs >= 0.5).astype(int)

cm = confusion_matrix(labels, preds)
tn, fp, fn, tp = cm.ravel()
parsbert_metrics = {
    "accuracy": round(float(accuracy_score(labels, preds)), 4),
    "precision": round(float(precision_score(labels, preds, zero_division=0)), 4),
    "recall": round(float(recall_score(labels, preds, zero_division=0)), 4),
    "f1Score": round(float(f1_score(labels, preds, zero_division=0)), 4),
}
parsbert_confusion = {
    "trueNegative": int(tn), "falsePositive": int(fp),
    "falseNegative": int(fn), "truePositive": int(tp),
}
print(parsbert_metrics)

In [ ]:
# 5d. Serving-contract check: rebuild the model exactly the way utils.py's
# _load_parsbert_model does (BertConfig.from_dict -> AutoModel.from_config, i.e.
# with no pretrained weights) and assert the state_dict keys still line up. Catches
# architecture drift here rather than on the first live request after deployment.
from transformers import BertConfig

servable = ParsBERTClassifier(
    AutoModel.from_config(BertConfig.from_dict(parsbert_backbone.config.to_dict())),
    parsbert_hidden_size,
)
drift = set(parsbert_model.state_dict()) ^ set(servable.state_dict())
assert not drift, f"state_dict does not match the serving class: {drift}"
print("Serving contract OK —", len(parsbert_model.state_dict()), "tensors")

In [ ]:
torch.save(
    {
        "state_dict": parsbert_model.state_dict(),
        "config": {
            "seq_len": PARSBERT_SEQ_LEN,
            "hidden_size": parsbert_hidden_size,
            # Full backbone config (layers, heads, vocab size, ...), so the server
            # rebuilds an identically-shaped BertModel purely from this checkpoint
            # — no pretrained download at inference time.
            "backbone_config": parsbert_backbone.config.to_dict(),
        },
        "metrics": parsbert_metrics,
    },
    os.path.join(OUT_DIR, "parsbertClassifierModel.pt"),
)
# Export the WordPiece tokenizer so the server loads it fully offline;
# utils.py prefers this folder over a Hub fetch when it's present.
parsbert_tokenizer.save_pretrained(os.path.join(OUT_DIR, "parsbert_tokenizer"))
print("ParsBERT artifacts written to", OUT_DIR)

## 6. JEV — the typed decision data

The dataset only carries a binary label, but JEV answers *three* typed questions,
so soft targets are distilled from the already-trained and deployed TF-IDF +
Logistic Regression pipeline — the exact preprocessing and style features
`utils.py` uses at serving time, which is why the deployed `vectorizer` and
`logregModel` have to come along to Drive.

Per row:

- **choice** — ground-truth one-hot blended 0.8/0.2 with the teacher's probability,
  clipped to $[0.02, 0.98]$ so no target is ever exactly certain.
- **score** — a discretized Gaussian over the 5 ordinal levels centred at
  $4 \times$ the blended AI probability. That's what teaches "between levels"
  answers instead of a hard argmax.
- **noul** — 1 where the teacher itself is uncertain (probability in
  $[0.15, 0.85]$): those are the only rows where a human reviewer genuinely adds
  information, which is precisely the question the head is supposed to answer.

This is the slowest section on Colab's free 2-vCPU tier; both caches below are
written to Drive so a re-run skips it.

In [ ]:
from shekar import Lemmatizer, SentenceTokenizer, WordTokenizer
from shekar.preprocessing import EmojiRemover, NonPersianRemover, PunctuationRemover

_preprocess_pipeline = None


def get_preprocess_pipeline():
    global _preprocess_pipeline
    if _preprocess_pipeline is None:
        _preprocess_pipeline = (
            EmojiRemover()
            | PunctuationRemover()
            | NonPersianRemover()
            | ShekarNormalizer()
            | Lemmatizer()
            | WordTokenizer()
        )
    return _preprocess_pipeline


def preprocess_persian_text(text: str) -> str:
    """utils.preprocess_persian_text — the classic-ML / BiLSTM pipeline."""
    text = re.sub(r"\n", " ", str(text))
    return " ".join(get_preprocess_pipeline()(text))


TOKEN_CACHE = os.path.join(DRIVE_DIR, "tokenized_texts.joblib")

try:
    tokenized = joblib.load(TOKEN_CACHE)
    assert len(tokenized) == len(df)
    print(f"Loaded cached tokenization for {len(tokenized)} documents.")
except (FileNotFoundError, AssertionError):
    print(f"Tokenizing {len(df)} documents on {multiprocessing.cpu_count()} cores "
          "(the slow part)...")
    tokenized = Parallel(n_jobs=multiprocessing.cpu_count())(
        delayed(preprocess_persian_text)(t) for t in df["text"]
    )
    joblib.dump(tokenized, TOKEN_CACHE)
    print("Cached to Drive for next time.")

# Only the rows JEV actually trains/evaluates on need preprocessing.
needed = np.concatenate([train_idx, test_idx])
cleaned = {int(i): " ".join(tokenized[i]) for i in needed}

In [ ]:
# Stylometric features for the JEV teacher. These formulas must stay byte-for-byte
# identical to utils.compute_style_features, including the 4-decimal rounding —
# the deployed logistic regression was fit on exactly these columns.
vectorizer = joblib.load(os.path.join(DRIVE_DIR, "tfidf_vectorizer.joblib"))
# Two project files look alike and only one of them is the vectorizer:
# tfidf_vectorizer.joblib is the TfidfVectorizer, tfidf_vectorizer_X.joblib is the
# 9.3M-column feature matrix it produces. Uploading the wrong one used to fail
# thousands of rows into the style-feature loop, so check it up front.
if not hasattr(vectorizer, "transform"):
    raise TypeError(
        f"{os.path.join(DRIVE_DIR, 'tfidf_vectorizer.joblib')} is a "
        f"{type(vectorizer).__name__}, not a TfidfVectorizer. You probably "
        "uploaded tfidf_vectorizer_X.joblib (the feature matrix) instead."
    )
STYLE_FEATURE_NAMES = ["entropy", "sentence_variety", "repetition", "coherence"]


def shannon_entropy(text):
    counts = Counter(text)
    total = len(text)
    entropy = 0.0
    for count in counts.values():
        p = count / total
        entropy -= p * math.log2(p)
    return entropy


def sentence_variety(text):
    sentences = list(SentenceTokenizer().tokenize(text))
    if len(sentences) < 2:
        return 0.0
    lengths = [len(s.split()) for s in sentences]
    mean_length = float(np.mean(lengths))
    return 0.0 if mean_length == 0 else min(float(np.std(lengths)) / mean_length, 1.0)


def pattern_repetition(text, n=3):
    tokens = list(WordTokenizer().tokenize(text))
    if len(tokens) < n:
        return 0.0
    counts = Counter(" ".join(g) for g in zip(*[tokens[i:] for i in range(n)]))
    return sum(1 for c in counts.values() if c > 1) / len(counts)


def text_coherence(text, vectorizer):
    sentences = [s for s in SentenceTokenizer().tokenize(text) if s.strip()]
    if len(sentences) < 2:
        return 1.0
    matrix = vectorizer.transform(sentences)
    sims = [
        float(cosine_similarity(matrix[i], matrix[i + 1])[0][0])
        for i in range(matrix.shape[0] - 1)
    ]
    return min(max(float(np.mean(sims)), 0.0), 1.0)


def compute_style_features(text, cleaned_text, vectorizer):
    entropy = shannon_entropy(text)
    max_entropy = math.log2(max(len(set(cleaned_text.replace(" ", ""))), 2))
    return {
        "entropy": round(min(entropy / max_entropy, 1.0), 4),
        "sentence_variety": round(sentence_variety(text), 4),
        "repetition": round(pattern_repetition(text, n=3), 4),
        "coherence": round(text_coherence(text, vectorizer), 4),
    }

In [ ]:
teacher = joblib.load(os.path.join(DRIVE_DIR, "logregModel.joblib"))

STYLE_CACHE = os.path.join(DRIVE_DIR, "jev_style_features.joblib")
try:
    style_matrix = joblib.load(STYLE_CACHE)
    assert len(style_matrix) == len(needed)
    print(f"Loaded cached style features for {len(style_matrix)} rows.")
except (FileNotFoundError, AssertionError):
    print(f"Computing style features for {len(needed)} rows (single-process, "
          "the vectorizer is shared)...")
    style_matrix = np.array([
        [compute_style_features(str(df["text"].iloc[i]), cleaned[int(i)], vectorizer)[name]
         for name in STYLE_FEATURE_NAMES]
        for i in tqdm(needed, desc="style features")
    ])
    joblib.dump(style_matrix, STYLE_CACHE)
    print("Cached to Drive for next time.")

tfidf = vectorizer.transform([cleaned[int(i)] for i in needed])
teacher_features = hstack([tfidf, csr_matrix(style_matrix)]).tocsr()
teacher_prob = teacher.predict_proba(teacher_features)[:, 1]
print(f"Teacher scored {len(needed)} rows; mean P(AI) = {teacher_prob.mean():.4f}")

In [ ]:
# JEV typed-question schema. Keep in sync with training/train_jev.py and with the
# answer rendering in utils.py.
CHOICE_CANDIDATES = ["human_written", "ai_generated"]
SCORE_LEVELS = [
    "definitely_human",
    "probably_human",
    "mixed_signals",
    "probably_ai",
    "definitely_ai",
]
QUESTIONS = {
    "source": {"type": "choice", "candidates": CHOICE_CANDIDATES},
    "ai_level": {"type": "score", "levels": SCORE_LEVELS},
    "needs_review": {"type": "noul"},
}

LABEL_WEIGHT = 0.8      # ground-truth one-hot weight in the choice target
TEACHER_WEIGHT = 0.2   # distillation weight from the teacher's probability
CHOICE_CLIP = (0.02, 0.98)
SCORE_SIGMA = 0.9
NOUL_BAND = (0.15, 0.85)

PROCEED_THRESHOLD = 0.8
NEEDS_REVIEW_THRESHOLD = 0.6


def choice_target(label, teacher_prob):
    t_ai = LABEL_WEIGHT * float(label) + TEACHER_WEIGHT * float(teacher_prob)
    t_ai = min(max(t_ai, CHOICE_CLIP[0]), CHOICE_CLIP[1])
    return {"human_written": round(1.0 - t_ai, 4), "ai_generated": round(t_ai, 4)}, t_ai


def score_target(t_ai):
    center = (len(SCORE_LEVELS) - 1) * t_ai
    levels = np.arange(len(SCORE_LEVELS), dtype=float)
    probs = np.exp(-((levels - center) ** 2) / (2.0 * SCORE_SIGMA**2))
    probs /= probs.sum()
    return [round(float(p), 4) for p in probs]


def noul_target(teacher_prob):
    return 1.0 if NOUL_BAND[0] <= teacher_prob <= NOUL_BAND[1] else 0.0


def build_records(indices, split_name, position):
    records = []
    for k, i in enumerate(indices):
        label = int(y_all[i])
        source_t, t_ai = choice_target(label, teacher_prob[position[k]])
        records.append({
            "id": f"{split_name}-{i:06d}",
            "split": split_name,
            "label": label,
            "text": str(df["text"].iloc[i]),
            "targets": {
                "source": source_t,
                "ai_level": score_target(t_ai),
                "needs_review": noul_target(teacher_prob[position[k]]),
            },
            "teacherAiProbability": round(float(teacher_prob[position[k]]), 4),
        })
    return records


train_val_records = build_records(train_idx, "train", range(len(train_idx)))
test_records = build_records(
    test_idx, "test", range(len(train_idx), len(train_idx) + len(test_idx))
)

# Carve the validation slice out of the training records, same stratified rule.
train_positions, val_positions = train_test_split(
    np.arange(len(train_val_records)),
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=[r["label"] for r in train_val_records],
)
train_records = [train_val_records[p] for p in train_positions]
val_records = [train_val_records[p] for p in val_positions]

for name, recs in [("train", train_records), ("val", val_records), ("test", test_records)]:
    noul_pos = sum(r["targets"]["needs_review"] for r in recs)
    print(f"{name}: {len(recs)} records | needs_review targets={int(noul_pos)}")

print(json.dumps(
    {**train_records[0], "text": train_records[0]["text"][:100] + "..."},
    ensure_ascii=False, indent=2,
))

## 7. JEV — the typed decision model

A GPT-2 backbone pooled on the **last real token** (causal attention, so only the
final position has seen everything) feeding three small linear heads, all answered
in one forward pass instead of autoregressive generation:

- **loss** — weighted sum of soft cross-entropy on the choice and score
  distributions plus binary cross-entropy on the noul head, where the "targets" are
  the teacher's distributions from the previous section rather than hard labels.

Only the last 4 of 12 transformer blocks plus the final LayerNorm are unfrozen,
same as the local trainer: 21k rows is not enough to move a pretrained model much,
and full fine-tuning would overfit fast.

The class must stay identical to `utils.JevDecisionModel`; section 7d checks it.

In [ ]:
from transformers import GPT2LMHeadModel

GPT2_MODEL_NAME = "HooshvareLab/gpt2-fa"

gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = "<pad>"

gpt2_lm = GPT2LMHeadModel.from_pretrained(GPT2_MODEL_NAME)
jev_backbone = gpt2_lm.transformer
jev_hidden_size = jev_backbone.config.n_embd
jev_n_layer = jev_backbone.config.n_layer
print(f"fast tokenizer: {gpt2_tokenizer.is_fast} | n_layer={jev_n_layer} "
      f"n_embd={jev_hidden_size}")

JEV_SEQ_LEN = 128
JEV_BATCH_SIZE = 32
JEV_MAX_EPOCHS = 10
JEV_PATIENCE = 2
JEV_LR_BACKBONE = 2e-5
JEV_LR_HEAD = 1e-3
JEV_UNFROZEN_BLOCKS = 4
DROPOUT = 0.1

CHOICE_WEIGHT, SCORE_WEIGHT, NOUL_WEIGHT = 1.0, 0.5, 0.5

for p in jev_backbone.parameters():
    p.requires_grad = False
for block in jev_backbone.h[jev_n_layer - JEV_UNFROZEN_BLOCKS:]:
    for p in block.parameters():
        p.requires_grad = True
for p in jev_backbone.ln_f.parameters():
    p.requires_grad = True


class JevDecisionModel(nn.Module):
    # Must stay architecturally identical to utils.py's JevDecisionModel.

    def __init__(self, backbone, hidden_size, n_choice, n_score_levels):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(DROPOUT)
        self.choice_head = nn.Linear(hidden_size, n_choice)
        self.score_head = nn.Linear(hidden_size, n_score_levels)
        self.noul_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        last_idx = attention_mask.sum(dim=1) - 1
        pooled = self.dropout(hidden[torch.arange(hidden.size(0)), last_idx])
        return (
            self.choice_head(pooled),
            self.score_head(pooled),
            self.noul_head(pooled).squeeze(-1),
        )


jev_model = JevDecisionModel(
    jev_backbone, jev_hidden_size, n_choice=len(CHOICE_CANDIDATES),
    n_score_levels=len(SCORE_LEVELS),
).to(DEVICE)

jev_optimizer = torch.optim.AdamW([
    {"params": [p for p in jev_backbone.parameters() if p.requires_grad], "lr": JEV_LR_BACKBONE},
    {"params": jev_model.choice_head.parameters(), "lr": JEV_LR_HEAD},
    {"params": jev_model.score_head.parameters(), "lr": JEV_LR_HEAD},
    {"params": jev_model.noul_head.parameters(), "lr": JEV_LR_HEAD},
])
print("JEV model ready.")

In [ ]:
def jev_encode(text_batch):
    enc = gpt2_tokenizer(
        list(text_batch),
        return_tensors="pt",
        truncation=True,
        max_length=JEV_SEQ_LEN,
        padding="max_length",
    )
    return enc["input_ids"], enc["attention_mask"]


def jev_collate(batch):
    texts, targets, labels = zip(*batch)
    input_ids, attention_mask = jev_encode(texts)
    return (
        input_ids,
        attention_mask,
        torch.tensor(
            [[t["source"][c] for c in CHOICE_CANDIDATES] for t in targets],
            dtype=torch.float32,
        ),
        torch.tensor([t["ai_level"] for t in targets], dtype=torch.float32),
        torch.tensor([t["needs_review"] for t in targets], dtype=torch.float32),
        torch.tensor(labels, dtype=torch.float32),
    )


def to_jev_dataset(records):
    return [(r["text"], r["targets"], r["label"]) for r in records]


jev_train_loader = DataLoader(
    to_jev_dataset(train_records), batch_size=JEV_BATCH_SIZE, shuffle=True, collate_fn=jev_collate,
)
jev_val_loader = DataLoader(
    to_jev_dataset(val_records), batch_size=64, shuffle=False, collate_fn=jev_collate,
)
jev_test_loader = DataLoader(
    to_jev_dataset(test_records), batch_size=64, shuffle=False, collate_fn=jev_collate,
)


def soft_cross_entropy(logits, target_probs):
    return -(target_probs * torch.log_softmax(logits, dim=-1)).sum(dim=-1).mean()


def jev_loss(choice_logits, score_logits, noul_logit, choice_t, score_t, noul_t):
    return (
        CHOICE_WEIGHT * soft_cross_entropy(choice_logits, choice_t)
        + SCORE_WEIGHT * soft_cross_entropy(score_logits, score_t)
        + NOUL_WEIGHT * nn.functional.binary_cross_entropy_with_logits(noul_logit, noul_t)
    )


@torch.no_grad()
def jev_eval(loader):
    jev_model.eval()
    choice_p, score_p, noul_p, labels = [], [], [], []
    for input_ids, attention_mask, _, _, _, yb in loader:
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            cl, sl, nl = jev_model(input_ids.to(DEVICE), attention_mask.to(DEVICE))
        choice_p.append(torch.softmax(cl, dim=-1).float().cpu().numpy())
        score_p.append(torch.softmax(sl, dim=-1).float().cpu().numpy())
        noul_p.append(torch.sigmoid(nl).float().cpu().numpy())
        labels.append(yb.numpy())
    return (np.concatenate(choice_p), np.concatenate(score_p),
            np.concatenate(noul_p), np.concatenate(labels))

In [ ]:
best_val_f1, best_state, stale = -1.0, None, 0

for epoch in range(1, JEV_MAX_EPOCHS + 1):
    t0 = time.time()
    jev_model.train()
    total_loss = 0.0
    for i, batch in enumerate(jev_train_loader, start=1):
        input_ids, attention_mask, choice_t, score_t, noul_t, _ = [
            t.to(DEVICE) for t in batch
        ]
        jev_optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            cl, sl, nl = jev_model(input_ids, attention_mask)
            loss = jev_loss(cl, sl, nl, choice_t, score_t, noul_t)
        scaler.scale(loss).backward()
        scaler.unscale_(jev_optimizer)
        torch.nn.utils.clip_grad_norm_(jev_model.parameters(), max_norm=1.0)
        scaler.step(jev_optimizer)
        scaler.update()
        total_loss += loss.item() * len(input_ids)
        if i % 50 == 0:
            print(f"  epoch {epoch} batch {i}/{len(jev_train_loader)} "
                  f"loss={loss.item():.4f} elapsed={time.time() - t0:.0f}s", flush=True)

    val_choice_p, _, _, val_labels = jev_eval(jev_val_loader)
    val_f1 = f1_score(val_labels, val_choice_p.argmax(axis=1))
    print(f"Epoch {epoch}/{JEV_MAX_EPOCHS}  loss={total_loss / len(train_records):.4f}  "
          f"val_f1={val_f1:.4f}  ({time.time() - t0:.0f}s)")

    if val_f1 > best_val_f1:
        best_val_f1, stale = val_f1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in jev_model.state_dict().items()}
    else:
        stale += 1
        if stale >= JEV_PATIENCE:
            print(f"Early stopping at epoch {epoch} (no val improvement for {JEV_PATIENCE} epochs).")
            break

jev_model.load_state_dict(best_state)
print(f"Best validation F1: {best_val_f1:.4f}")

In [ ]:
def expected_calibration_error(probs, labels, n_bins=15):
    confidence = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)


choice_p, score_p, noul_p, test_labels = jev_eval(jev_test_loader)
preds = choice_p.argmax(axis=1)
cm = confusion_matrix(test_labels, preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

score_t_test = np.array([r["targets"]["ai_level"] for r in test_records])
noul_t_test = np.array([r["targets"]["needs_review"] for r in test_records])

jev_metrics = {
    "accuracy": round(float(accuracy_score(test_labels, preds)), 4),
    "precision": round(float(precision_score(test_labels, preds, zero_division=0)), 4),
    "recall": round(float(recall_score(test_labels, preds, zero_division=0)), 4),
    "f1Score": round(float(f1_score(test_labels, preds, zero_division=0)), 4),
    # Per-type answers: score-answer error against the teacher's 0..4 scale, noul
    # ranking quality, and calibration of the choice distribution.
    "scoreMae": round(float(mean_absolute_error(
        score_t_test @ np.arange(len(SCORE_LEVELS)),
        score_p @ np.arange(len(SCORE_LEVELS)),
    )), 4),
    "choiceEce": round(expected_calibration_error(choice_p, test_labels), 4),
}
# Both noul classes must be present for AUC; on a near-separable dataset the
# uncertain band can be nearly empty.
if len(np.unique(noul_t_test)) > 1:
    jev_metrics["noulAuc"] = round(float(roc_auc_score(noul_t_test, noul_p)), 4)
    jev_metrics["noulAveragePrecision"] = round(
        float(average_precision_score(noul_t_test, noul_p)), 4
    )

jev_confusion = {
    "trueNegative": int(tn), "falsePositive": int(fp),
    "falseNegative": int(fn), "truePositive": int(tp),
}
print(jev_metrics)

In [ ]:
# 7d. Serving-contract check. utils.py's _load_jev_model builds the backbone from
# the local gpt2-fa.mdl pickle and then loads this state_dict on top, so the two
# must expose the same tensor names. HF_MODEL_KEYS are the ones that file
# contributes; the rest are the three decision heads.
print(f"state_dict tensors: {len(jev_model.state_dict())}")
print(f"backbone tensors:   {len(jev_backbone.state_dict())} + 3 heads "
      f"(choice/score/noul)")
assert any(k.startswith("backbone.h.11.") for k in jev_model.state_dict())
assert "choice_head.weight" in jev_model.state_dict()
assert "score_head.weight" in jev_model.state_dict()
assert "noul_head.weight" in jev_model.state_dict()
print("Serving contract OK.")

In [ ]:
torch.save(
    {
        "state_dict": jev_model.state_dict(),
        "config": {
            "seq_len": JEV_SEQ_LEN,
            "hidden_size": jev_hidden_size,
            "n_choice": len(CHOICE_CANDIDATES),
            "n_score_levels": len(SCORE_LEVELS),
            "choice_candidates": CHOICE_CANDIDATES,
            "score_levels": SCORE_LEVELS,
            "thresholds": {
                "proceed": PROCEED_THRESHOLD,
                "needsReview": NEEDS_REVIEW_THRESHOLD,
            },
        },
        "metrics": jev_metrics,
    },
    os.path.join(OUT_DIR, "jevModel.pt"),
)
# utils.py prefers this folder over the old slow-tokenizer pickle when present, so
# serving never needs the Hub.
gpt2_tokenizer.save_pretrained(os.path.join(OUT_DIR, "gpt2_tokenizer"))
print("JEV artifacts written to", OUT_DIR)

## 8. Summary

In [ ]:
for name, metrics in [("parsbert", parsbert_metrics), ("jev_system_one", jev_metrics)]:
    print(f"{name}: {metrics}")

## 9. Write the `models.json` fragment

Same schema and ids `models.json` already uses (`parsbert`, `jev_system_one`) and
the same `runtime` values `utils.py` dispatches on (`pytorch-parsbert`,
`pytorch-jev`), so merging this needs no backend change.

`ensembleWeight` is carried over from the current `models.json` when that file is
on Drive — the ensemble in `utils._ensemble_predict` skips any model weighted 0,
and silently changing that here would change every prediction the API returns.

In [ ]:
MODELS_JSON = os.path.join(DRIVE_DIR, "models.json")
existing = {}
if os.path.isfile(MODELS_JSON):
    with open(MODELS_JSON, encoding="utf-8") as f:
        existing = {m["id"]: m for m in json.load(f)}
else:
    print(f"No models.json in {DRIVE_DIR}; defaulting ensembleWeight to 1.0.")


def weight_for(model_id, default):
    return float(existing.get(model_id, {}).get("ensembleWeight", default))


n_total = len(train_records) + len(val_records) + len(test_records)

entries = [
    {
        "id": "parsbert",
        "name": "ParsBERT (Bidirectional Encoder)",
        "nameFa": "پارس‌برت (رمزگذار دوسویه)",
        "description": (
            "ParsBERT (HooshvareLab/bert-fa-base-uncased) fine-tuned end to end as "
            "a bidirectional sequence classifier with [CLS] pooling, on a Colab GPU "
            "over the full dataset with mixed precision and validation-based early "
            "stopping. Unlike the causal GPT-2 classifier, every token attends to "
            "the whole sequence and a dedicated [CLS] position summarizes it."
        ),
        "descriptionFa": (
            "پارس‌برت (HooshvareLab/bert-fa-base-uncased) که به‌طور کامل روی کل "
            "مجموعه داده و با استفاده از GPU، دقت مختلط و توقف زودهنگام مبتنی بر "
            "اعتبارسنجی، به‌عنوان یک دسته‌بند دوسویه فاین‌تیون شده است. برخلاف "
            "دسته‌بند علّی GPT-2، در اینجا هر توکن به کل توالی توجه می‌کند و توکن "
            "اختصاصی [CLS] خلاصهٔ توالی را نمایندگی می‌کند."
        ),
        "algorithm": "BertModel + linear head on [CLS]",
        "type": "dl",
        "version": "2.0",
        "filename": "parsbertClassifierModel.pt",
        "runtime": "pytorch-parsbert",
        "ensembleWeight": weight_for("parsbert", 1.0),
        "metrics": parsbert_metrics,
        "confusionMatrix": parsbert_confusion,
        "training": {
            "totalSamples": len(train_idx) + len(val_idx) + len(test_idx),
            "trainSamples": len(train_idx),
            "testSamples": len(test_idx),
            "featureCount": parsbert_hidden_size,
            "testSplitRatio": TEST_SIZE,
            "randomSeed": RANDOM_SEED,
            "dataset": "processedDataset.csv",
        },
    },
    {
        "id": "jev_system_one",
        "name": "JEV System One (Decision Head)",
        "nameFa": "مدل تصمیم سیستم-یک (JEV)",
        "description": (
            "JEV-style 'System One' decision model: the Persian GPT-2 backbone "
            "(HooshvareLab/gpt2-fa, last 4 blocks fine-tuned) pooled into three typed "
            "decision heads answered in a single forward pass -- a choice question "
            "(human vs AI, as a probability distribution), a score question "
            "(machine-likeness on a 5-level scale, answered as a probability-weighted "
            "value between levels) and a noul question (probability that human review "
            "is needed). Trained on soft targets distilled from the deployed TF-IDF + "
            "Logistic Regression pipeline."
        ),
        "descriptionFa": (
            "مدل تصمیم «سیستم-یک» به سبک JEV: بک‌بون GPT-2 فارسی "
            "(HooshvareLab/gpt2-fa با فاین‌تیون ۴ بلوک آخر) که بازنمایی تجمیع‌شده آن "
            "به سه سر تصمیم نوع‌بندی‌شده می‌رود و همه در یک گذر رو به جلو پاسخ داده "
            "می‌شوند — سؤال انتخاب (انسان در برابر هوش مصنوعی به شکل توزیع احتمال)، "
            "سؤال امتیاز (ماشین‌ماندبودن روی طیف ۵ سطحی با مقدار وزن‌دار احتمالاتی که "
            "می‌تواند بین سطوح بیفتد) و سؤانول (احتمال نیاز به بازبینی انسانی). "
            "آموزش با هدف‌های نرم تقطیرشده از خط لولهٔ TF-IDF + رگرسیون لجستیکِ مستقر."
        ),
        "algorithm": "GPT2 + typed decision heads (choice/score/noul)",
        "type": "dl",
        "version": "2.0",
        "filename": "jevModel.pt",
        "runtime": "pytorch-jev",
        "ensembleWeight": weight_for("jev_system_one", 1.0),
        "metrics": jev_metrics,
        "confusionMatrix": jev_confusion,
        "training": {
            "totalSamples": n_total,
            "trainSamples": len(train_records),
            "testSamples": len(test_records),
            "featureCount": jev_hidden_size,
            "testSplitRatio": len(test_records) / n_total,
            "randomSeed": RANDOM_SEED,
            "dataset": "processedDataset.csv (JEV records, teacher-distilled soft targets)",
        },
    },
]

with open(os.path.join(OUT_DIR, "models_colab_update.json"), "w", encoding="utf-8") as f:
    json.dump(entries, f, ensure_ascii=False, indent=2)
print("Wrote models_colab_update.json with ids:", [e["id"] for e in entries])

## 10. Zip and download

Everything is already on Drive. This bundles the same files for a direct browser
download as well, in case you'd rather pull them into the project from here.

To deploy: copy everything from `MyDrive/AITextDetection/trained/` into the project
root, merge `models_colab_update.json` into `models.json` (it reuses the existing
ids, so replace those two entries), and restart `server.py`. No backend code
changes — `utils.py` already dispatches both runtimes.

In [ ]:
STAGE_DIR = "/content/jev_parsbert_out"
if os.path.isdir(STAGE_DIR):
    shutil.rmtree(STAGE_DIR)
shutil.copytree(OUT_DIR, STAGE_DIR)

zip_path = "/content/jev_parsbert_models.zip"
shutil.make_archive(
    os.path.splitext(zip_path)[0], "zip", root_dir=STAGE_DIR
)

total_mb = 0
for root, _, files in os.walk(STAGE_DIR):
    for name in files:
        path = os.path.join(root, name)
        size_mb = os.path.getsize(path) / 1e6
        total_mb += size_mb
        print(f"{size_mb:9.1f} MB  {os.path.relpath(path, STAGE_DIR)}")
print(f"{total_mb:9.1f} MB  total")

from google.colab import files

files.download(zip_path)